# Run this cell once to absolutely guarantee the CPU-ONLY version of PyTorch 
# is installed in the EXACT Python environment this notebook is using.
import sys
!{sys.executable} -m pip uninstall -y torch torchvision torchaudio
!{sys.executable} -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

In [1]:
import os
import sys
import time
import numpy as np

# --- ENVIRONMENT SETUP ---
base_dir = os.getcwd()

# 1. Insert the local C: drive venv with HIGHEST priority FIRST
for venv_folder in ["venv", ".venv"]:
    site_packages = os.path.join(base_dir, venv_folder, "Lib", "site-packages")
    if os.path.exists(site_packages) and site_packages not in sys.path:
        sys.path.insert(0, site_packages)
        break

# 2. Import PyTorch NOW, before the Z: drive is anywhere in the path
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

print(f"PyTorch version {torch.__version__} loaded.")

# 3. Append the global Z: drive venv AFTER PyTorch has safely loaded its local DLLs
global_venv_site_packages = r"Z:\ai project\.venv\Lib\site-packages"
if os.path.exists(global_venv_site_packages) and global_venv_site_packages not in sys.path:
    sys.path.append(global_venv_site_packages)

try:
    from thrember.features import PEFeatureExtractor
    ndim = PEFeatureExtractor.dim
    print(f"Features extractor imported. Dimension: {ndim}")
except ImportError:
    ndim = 2381
    print(f"Warning: 'thrember' not found. Using default dimension: {ndim}")

OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "c:\Users\him\ember2024_project\.venv\lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

In [ ]:
# --- CONFIGURATION ---
DATASET_DIR = r"Z:\ember2024_train_data"
CHUNK_SIZE = 250000              # Safely pull 250k rows heavily into RAM at a time
BATCH_SIZE = 4096                # Process in parallel 4096 samples at a time
EPOCHS = 10                      # Total full passes over the dataset
LEARNING_RATE = 0.001
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using compute device: {DEVICE}")

In [ ]:
# --- MODEL ARCHITECTURE ---
class MalwareDNN(nn.Module):
    def __init__(self, input_dim):
        super(MalwareDNN, self).__init__()
        
        self.net = nn.Sequential(
            nn.Linear(input_dim, 2048),
            nn.BatchNorm1d(2048),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(2048, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.1),
            
            nn.Linear(128, 1) 
            # No Sigmoid here! We use BCEWithLogitsLoss for numeric stability
        )

    def forward(self, x):
        return self.net(x)

### Setup Data Streaming & Initialization
Here we initialize the memory mapping so we can loop over the data gracefully.

In [ ]:
X_path = os.path.join(DATASET_DIR, "X_train.dat")
y_path = os.path.join(DATASET_DIR, "y_train.dat")

if not os.path.exists(X_path) or not os.path.exists(y_path):
    raise FileNotFoundError("Training data files not found.")

file_size = os.path.getsize(X_path)
nrows = file_size // (ndim * 4)
train_nrows = int(nrows * 0.9)  # Reserve last 10% for test

# Open Memory map
X_memmap = np.memmap(X_path, dtype=np.float32, mode="r", shape=(nrows, ndim))
y_memmap = np.memmap(y_path, dtype=np.int32, mode="r", shape=(nrows,))

print(f"Total Rows: {nrows:,} | Training on {train_nrows:,} samples.")

### The Training & Checkpoint Loop
This loop streams the chunks, batches the data to the GPU/CPU, and actively records progress to `ember_dnn_checkpoint.pth`. If interrupted, re-running this cell seamlessly recovers the exact epoch and chunk index.

In [ ]:
model = MalwareDNN(input_dim=ndim).to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=1)

checkpoint_path = os.path.join(DATASET_DIR, "ember_dnn_checkpoint.pth")

start_epoch = 0
start_chunk_idx = 0

# --- SMART RESUME (CHECKPOINT LOGIC) ---
if os.path.exists(checkpoint_path):
    print(f"Found existing checkpoint at {checkpoint_path}...")
    try:
        checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        start_epoch = checkpoint['epoch']
        start_chunk_idx = checkpoint['chunk_idx']
        print(f"✅ Successfully Resumed from Epoch {start_epoch + 1}, Chunk {start_chunk_idx}")
    except Exception as e:
        print(f"⚠️ Error loading checkpoint: {e}. Starting fresh.")
else:
    print("No checkpoint found. Starting fresh training.")

# --- COMPUTE CHUNK METRICS ---
total_chunks = (train_nrows + CHUNK_SIZE - 1) // CHUNK_SIZE

print("\nStarting Out-of-Core Batch Training...")

# --- MAIN LOOP ---
for epoch in range(start_epoch, EPOCHS):
    model.train()
    epoch_loss = 0.0
    processed_samples = 0
    start_time = time.time()
    
    current_chunk = 0
    
    # Stream data in massive chunks to avoid RAM crashes
    for start_idx in range(0, train_nrows, CHUNK_SIZE):
        
        # Skip previously trained chunks if resuming mid-epoch
        if epoch == start_epoch and current_chunk < start_chunk_idx:
            current_chunk += 1
            continue
            
        end_idx = min(start_idx + CHUNK_SIZE, train_nrows)
        
        # Load exact chunk into RAM
        X_chunk = np.array(X_memmap[start_idx:end_idx])
        y_chunk = np.array(y_memmap[start_idx:end_idx])
        
        # Filter unlabeled
        valid_idx = y_chunk != -1
        X_chunk, y_chunk = X_chunk[valid_idx], y_chunk[valid_idx]
        
        if len(y_chunk) > 0:
            # Convert to PyTorch tensors
            X_tensor = torch.tensor(X_chunk, dtype=torch.float32)
            y_tensor = torch.tensor(y_chunk, dtype=torch.float32).unsqueeze(1) # [batch_size, 1]
            
            # Create DataLoader for this specific RAM chunk
            dataset = TensorDataset(X_tensor, y_tensor)
            dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
            
            # Mini-batch training
            for batch_X, batch_y in dataloader:
                batch_X, batch_y = batch_X.to(DEVICE), batch_y.to(DEVICE)
                
                optimizer.zero_grad()
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
                
                epoch_loss += loss.item() * batch_X.size(0)
                processed_samples += batch_X.size(0)
                
        print(f"  [E{epoch+1}] Chunk {current_chunk}/{total_chunks}: Processed up to row {end_idx:,}")
        current_chunk += 1
        
        # SAVE CHECKPOINT AFTER EVERY CHUNK (Crash Protection)
        torch.save({
            'epoch': epoch,
            'chunk_idx': current_chunk, # Next chunk to process
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'loss': loss.item() if 'loss' in locals() else 0.0
        }, checkpoint_path)

    # After full epoch completes: Reset chunk index
    start_chunk_idx = 0 
    
    avg_loss = epoch_loss / max(1, processed_samples)
    elapsed = time.time() - start_time
    print(f"=> EPOCH {epoch+1} COMPLETE | Avg Loss: {avg_loss:.4f} | Time: {elapsed:.2f} sec\n")
    
    # Step learning rate scheduler
    scheduler.step(avg_loss)
    
print("\n🎉 Deep Learning Training Complete! Final model is safely at 'ember_dnn_checkpoint.pth'.")